[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Optimizer077/jev-learning-course/blob/main/notebooks/08_exercises_and_solutions.ipynb)

# 8 · Worked exercises and explanations

**Path:** Practice and review  
**Before you start:** Attempt the matching exercise before opening its answer

[Course home](../README.md) · [Course outline](../docs/COURSE.md) · [Setup help](../docs/SETUP.md) · [Glossary](../docs/GLOSSARY.md)

## The idea before the code

A useful explanation lets you predict what will happen when something changes. These worked answers check that skill.

**Your first pass:** Read only the solution you need. Return to the originating lesson after checking it.

## Goal
Check whether you can reason about the output contract, the math, and the software policy.
Try each exercise in its lesson before reading this notebook. These are worked solutions, not a graded test.

## Setup
This notebook is independent of previous kernels and makes no API calls.

In [1]:
# Find the included teaching helpers from the course folder.
from pathlib import Path
import sys
REQUIRED_FILES = ("src/lab_core.py", "src/tutorial_utils.py", "src/torch_lab.py",
                  "data/tickets.json", "data/torch_toy.json",
                  "assets/decision-flow.png", "assets/question-types.png",
                  "assets/calibration-counts.png", "assets/data-splits.png", "assets/word-order.png")
COURSE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if all((p / name).is_file() for name in REQUIRED_FILES)), None)
# Colab opens a single notebook; fetch its companion code and fictional data.
if COURSE_ROOT is None:
    try:
        import google.colab
    except ImportError:
        pass
    else:
        import subprocess
        COURSE_ROOT = Path("/content/jev-learning-course")
        if COURSE_ROOT.exists() and not all((COURSE_ROOT / name).is_file() for name in REQUIRED_FILES):
            raise FileNotFoundError("The cached Colab course is incomplete. Start a fresh runtime and run all again.")
        if not COURSE_ROOT.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/Optimizer077/jev-learning-course.git",
                            str(COURSE_ROOT)], check=True)
if COURSE_ROOT is None:
    raise FileNotFoundError("Extract the complete course and open it from its folder.")
if str(COURSE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from tutorial_utils import setup, show_figure, table, BLUE, ORANGE, GREY
setup()

from lab_core import softmax

## Steps
### 1. Choose a primitive
One queue → **Choice**. Explicit refund request → **Noul**. Ordered disruption levels → **Score**.
Writing an email requires generation; a bounded decision interface can choose a template or route
to a generative model, but cannot directly supply arbitrary new prose.

### 2. Shift and scale logits
Adding the same constant to every score cancels between numerator and denominator in softmax.
Multiplying by a positive constant preserves ranking, but changes concentration. A negative
multiplier would reverse ranking; multiplying by zero gives a uniform distribution.

In [2]:
z = np.array([.2, 1.4, -.3])
assert np.allclose(softmax(z), softmax(z+100))
assert softmax(z).argmax() == softmax(3*z).argmax()
assert not np.allclose(softmax(z), softmax(3*z))
table(['Transformation', 'Probability vector'], [('original',softmax(z).round(3)),
      ('add 100',softmax(z+100).round(3)),('multiply by 3',softmax(3*z).round(3))])

Transformation,Probability vector
original,[0.203 0.674 0.123]
add 100,[0.203 0.674 0.123]
multiply by 3,[0.026 0.968 0.006]


### 3. Derive the review region
“No” costs $p$, so it ties review at $p=0.2$. “Yes” costs $4(1-p)$, so it ties review at $p=0.95$.
Thus choose no below 0.2, review between 0.2 and 0.95, and yes above 0.95. At boundaries either tied
action has equal modeled cost. If review costs 0.8, the review region disappears except a three-way
tie at $p=0.8$: the best binary action already costs at most 0.8.

In [3]:
def cheapest_action(p, review_cost):
    values = {'yes':4*(1-p),'no':p,'review':review_cost}
    return min(values,key=values.get)
table(['p','Review costs 0.2','Review costs 0.8'],
      [(p,cheapest_action(p,.2),cheapest_action(p,.8)) for p in [.1,.5,.9,.99]])
assert cheapest_action(.5,.2)=='review'

p,Review costs 0.2,Review costs 0.8
0.1,no,no
0.5,review,no
0.9,review,yes
0.99,yes,yes


### 4. Confidence is not authorization
A high topic probability is irrelevant to a required permission check. The application must enforce
the permission deterministically before action. The model can supply a recommendation, while code
enforces which recommendations are allowed to be acted on.

In [4]:
def may_act(topic_probability, has_permission):
    return has_permission and topic_probability >= .9
assert may_act(.96,False) is False
assert may_act(.96,True) is True

### 5. Avoid label leakage
The `label` field is the desired answer. Supplying it to the model lets the model copy the reference
instead of inferring it. A legitimate policy describes general rules available at deployment time;
it does not reveal this example's hidden answer. Restrict state to an explicit allowlist of input fields.

In [5]:
row = {'text':'Please reset my password.','label':'account','split':'test'}
safe_state = {'customer_message':row['text']}
assert set(safe_state)=={'customer_message'}
print(safe_state)

{'customer_message': 'Please reset my password.'}


### 6. Explain the attention isolation test
Unblocking A→B lets A use B's key/value vectors, so perturbing B can change A. The equality assertion
will generally fail. “Parallel” describes scheduling; “isolated” describes allowed information flow.
State also must not absorb question information if it is reused across layers.

### 7. Work-sharing limit
The ratio is $m(S+Q)/(S+mQ)$. As $m$ tends to infinity, it approaches $(S+Q)/Q$.
For $S=100$, $Q=8$, the limit is 13.5. Memory bandwidth, latency, scheduling, and other overheads
are absent, so this does not establish a real model's speedup.

In [6]:
S,Q = 100,8
print('Limit of the assumed work ratio:',(S+Q)/Q)
print('Ratio at 16 questions:',16*(S+Q)/(S+16*Q))

Limit of the assumed work ratio: 13.5
Ratio at 16 questions: 7.578947368421052


### 8. An all-unknown input
With no recognized words, the feature vector is zero. The output is softmax of the bias alone.
Nothing in that operation detects that the request is out of domain.

In [7]:
from lab_core import train_text_baseline, vectorise
_,vocab,weights,bias,_ = train_text_baseline()
unknown = vectorise(['quasar nebula supernova'],vocab)
assert unknown.sum()==0
assert np.allclose(softmax(unknown@weights+bias),softmax(bias))
print('Probabilities with zero recognized features:',softmax(bias).round(3))

Probabilities with zero recognized features: [0.328 0.315 0.358]


### 9. Bigram extension: one possible implementation
Append adjacent token pairs to the unigram features. This gives “not locked” a different feature
from “locked,” but only training examples can teach the model what that feature means.
Sparse bigrams do not solve language understanding or generalize automatically to all paraphrases.

In [8]:
from lab_core import tokenise
def unigram_and_bigram_features(text):
    tokens = tokenise(text)
    return tokens + [a+'__'+b for a,b in zip(tokens,tokens[1:])]
print(unigram_and_bigram_features('My account is not locked'))
assert 'not__locked' in unigram_and_bigram_features('My account is not locked')

['my', 'account', 'is', 'not', 'locked', 'my__account', 'account__is', 'is__not', 'not__locked']


## Checks
You should now be able to explain why each of these implications is false:

1. Valid schema ⇒ correct answer.
2. Higher probability ⇒ permission to act.
3. Good accuracy ⇒ good calibration.
4. Parallel questions ⇒ independent events.
5. One successful API call ⇒ production readiness.
6. A generic attention demo ⇒ Jev's published architecture.

## Next Steps
Pick your own bounded task, write the answer space, collect labeled examples, and establish a
simple baseline before increasing complexity. Keep test examples held out. Use real Jev calls only
when you want to measure Jev, and keep the model's predictions separate from your operating policy.

[Return to the course map](00_start_here.ipynb).

## Take three ideas with you

- Predict, run, explain, and state a limitation.
- Keep output validity, accuracy, calibration, and authorization distinct.
- Use the smallest model and workflow that you can evaluate for your task.

**You are ready to move on when:** You can justify an answer, not just reproduce a number.

## Check your understanding

I can rerun every cell. Does that mean I understand the course?

Pause and explain your answer before opening the explanation.

<details>
<summary>Show the explanation</summary>
<p>A stronger check is to predict a small change, explain why it happens, and name one limitation of the experiment.</p>
</details>

For a short next step, try the [practice questions](../docs/PRACTICE.md).
For runnable exercises, use the [worked exercises](08_exercises_and_solutions.ipynb).

If you are unsure where to go next, use [the course outline](../docs/COURSE.md).

[Assignments](../assignments/README.md) · [Quick reference](../docs/QUICK_REFERENCE.md) · [Course outline](../docs/COURSE.md)